Nama: Roland Albertian Sehapikang<br>
Kelas: IF403<br>
Nim: 240401010294

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
jumlah_baris = df.shape[0]
jumlah_kolom = df.shape[1]
daftar_kolom = df.columns.tolist()

print(f"Jumlah baris: {jumlah_baris}")
print(f"Jumlah kolom: {jumlah_kolom}")
print(f"Daftar kolom: {daftar_kolom}")

Jumlah baris: 891
Jumlah kolom: 12
Daftar kolom: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


**Temuan:** Dataset Titanic ini terdiri dari 891 baris (observasi penumpang) dan 12 kolom (variabel). Kolom-kolom yang tersedia memuat informasi seperti ID Penumpang, status keselamatan (Survived), kelas tiket (Pclass), nama, jenis kelamin (Sex), usia (Age), hingga harga tiket (Fare) dan pelabuhan keberangkatan (Embarked).

In [3]:
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
missing_props = (missing_cols / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing': missing_cols,
    'Proporsi (%)': missing_props
})
print(missing_df)

          Jumlah Missing  Proporsi (%)
Age                  177         19.87
Cabin                687         77.10
Embarked               2          0.22


**Temuan:** Terdapat tiga kolom yang memiliki missing values (data kosong):
1. **Age (Usia):** Hilang 177 data (sekitar 19.87%). Ini cukup banyak dan memerlukan imputasi (misalnya diisi dengan median usia) sebelum pemodelan lebih lanjut.

2. **Cabin (Kabin):** Hilang 687 data (77.10%). Karena proporsinya yang kosong sangat besar (di atas 40-60%), kolom ini idealnya di-drop atau diabaikan dalam analisis.

3. **Embarked (Pelabuhan asal):** Hilang 2 data (0.22%). Karena sangat sedikit, kita bisa menghapus dua baris ini atau mengisinya dengan modus (nilai yang paling sering muncul).

In [4]:
persen_selamat = (df['Survived'].mean() * 100).round(2)
print(f"Persentase penumpang yang selamat: {persen_selamat}%")

Persentase penumpang yang selamat: 38.38%


**Temuan:** Hanya sekitar 38.38% penumpang yang selamat dari tragedi Titanic. Hal ini menunjukkan bahwa dataset ini agak imbalanced (tidak seimbang) karena mayoritas data (sekitar 61.62%) mewakili penumpang yang tidak selamat.

In [5]:
wanita_kelas1 = df[(df['Sex'] == 'female') & (df['Pclass'] == 1)]

jumlah_wanita_kelas1 = len(wanita_kelas1)
rata_usia_wanita_kelas1 = wanita_kelas1['Age'].mean().round(2)

print(f"Jumlah penumpang wanita di Kelas 1: {jumlah_wanita_kelas1} orang")
print(f"Rata-rata usia mereka: {rata_usia_wanita_kelas1} tahun")

Jumlah penumpang wanita di Kelas 1: 94 orang
Rata-rata usia mereka: 34.61 tahun


**Temuan:** Terdapat 94 penumpang wanita yang bepergian di tiket Kelas 1. Rata-rata usia mereka adalah sekitar 34.61 tahun. Operasi pemfilteran majemuk (&) ini sangat berguna untuk membuat subset data yang spesifik.

In [6]:
survival_per_class = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)

print("Tingkat Keselamatan Berdasarkan Kelas Tiket (%):")
print(survival_per_class)

kelas_tertinggi = survival_per_class.idxmax()
persentase_tertinggi = survival_per_class.max()

print(f"\nKelas dengan tingkat keselamatan paling tinggi adalah Kelas {kelas_tertinggi} ({persentase_tertinggi}%).")

Tingkat Keselamatan Berdasarkan Kelas Tiket (%):
Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64

Kelas dengan tingkat keselamatan paling tinggi adalah Kelas 1 (62.96%).


**Temuan:** Penumpang di Kelas 1 (Pclass=1) memiliki tingkat keselamatan tertinggi (62.96%), disusul Kelas 2 (47.28%), dan yang terendah adalah Kelas 3 (24.24%). Ini mengindikasikan adanya korelasi kuat antara status sosial/kelas tiket dengan prioritas evakuasi.

In [7]:
survival_per_sex = (df.groupby('Sex')['Survived'].mean() * 100).round(2)

print("Tingkat Keselamatan Berdasarkan Jenis Kelamin (%):")
print(survival_per_sex)

Tingkat Keselamatan Berdasarkan Jenis Kelamin (%):
Sex
female    74.20
male      18.89
Name: Survived, dtype: float64


**Temuan & Interpretasi:** Analisis groupby tambahan berdasarkan jenis kelamin (Sex) menunjukkan ketimpangan keselamatan yang sangat mencolok. Sebanyak 74.20% penumpang wanita berhasil selamat, dibandingkan dengan pria yang hanya mencatatkan angka keselamatan sebesar 18.89%.

Hal ini membuktikan secara data historis bahwa protokol maritim evakuasi "Women and children first" benar-benar diterapkan secara ketat selama peristiwa tenggelamnya Titanic. Kolom Sex akan menjadi salah satu fitur paling prediktif (memiliki Feature Importance tinggi) jika kita membangun model Machine Learning ke depannya.

# **Kesimpulan Singkat:**

**1. Apa yang dipelajari?**
* **Manipulasi Data dengan Pandas:** Mempelajari cara mengimpor dataset eksternal (CSV) dari internet menggunakan pd.read_csv() dan mengubahnya menjadi struktur DataFrame.
* **Inspeksi Data Dasar:** Menggunakan metode dan atribut dasar Pandas seperti .head(), .shape, dan .columns untuk memahami ukuran dan struktur awal data.
* **Deteksi Missing Values:** Memahami cara mengidentifikasi data yang kosong (NaN) di dalam dataset menggunakan .isnull().sum().
* **Pemfilteran & Agregasi:** Menguasai teknik seleksi data berdasarkan kondisi tertentu (boolean indexing) dan menggunakan .groupby() untuk menghitung statistik agregat (seperti rata-rata keselamatan per kelas atau gender).

**2. Temuan Utama**
* **Kondisi Dataset:** Dataset Titanic berisi 891 baris dan 12 kolom, namun kondisinya belum sepenuhnya bersih. Terdapat banyak missing values pada kolom Age (usia) dan Cabin (kabin).
* **Fakta Keselamatan (Survival):** Tragedi ini memiliki tingkat keselamatan yang rendah secara keseluruhan (hanya ~38.38% yang selamat).
* **Korelasi Status & Gender:** Analisis agregat membuktikan bahwa prioritas penyelamatan sangat dipengaruhi oleh kelas sosial dan gender. Penumpang Kelas 1 dan penumpang wanita memiliki persentase keselamatan yang jauh lebih tinggi dibandingkan penumpang Kelas 3 dan pria.

**3. Keterbatasan atau Pertanyaan yang muncul**
* Bagaimana cara terbaik untuk mengisi data usia (Age) yang kosong agar dataset ini siap digunakan untuk Machine Learning?
* Bagaimana cara memvisualisasikan temuan ini (misalnya grafik batang untuk tingkat keselamatan) agar lebih mudah dipahami secara visual tanpa harus membaca deretan angka?
* Bisakah kita menggabungkan beberapa kriteria sekaligus (misalnya wanita di kelas 3 dibandingkan pria di kelas 1) untuk analisis yang lebih dalam?